In [3]:
import os
import gc
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Librerías del benchmark
import datashader as ds
import vaex
import duckdb
import polars as pl
from sklearn.preprocessing import KBinsDiscretizer

# 1. Validación de Ruta y Carga del Dataset Original
RUTA = "../data/dataset_5M.parquet"
df = pd.read_parquet(RUTA)

print(f"Dataset cargado correctamente:")
print(f" - Filas: {df.shape[0]:,}")
print(f" - Columnas: {df.shape[1]}")

# 2. Creación de directorios para guardar resultados
os.makedirs("../resultados/tablas", exist_ok=True)
os.makedirs("../resultados/graficos", exist_ok=True)
os.makedirs("../resultados/representaciones", exist_ok=True)

print("\nDirectorios preparados en ../resultados/")

Dataset cargado correctamente:
 - Filas: 5,000,000
 - Columnas: 5

Directorios preparados en ../resultados/


In [4]:
# 1. Medición de tiempo de lectura (3 repeticiones y mediana)
tiempos_lectura = []
for i in range(3):
    inicio = time.perf_counter()
    temporal = pd.read_parquet(RUTA)
    fin = time.perf_counter()
    tiempos_lectura.append(fin - inicio)
    del temporal
    gc.collect()

mediana_lectura = np.median(tiempos_lectura)
print(f"Tiempos de lectura (s): {[round(t, 4) for t in tiempos_lectura]}")
print(f"Mediana tiempo de lectura: {mediana_lectura:.4f} segundos")

# 2. Medición de tamaño en disco vs memoria RAM en Pandas
tamaño_archivo_mb = os.path.getsize(RUTA) / (1024 ** 2)
memoria_dataframe_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"\nTamaño del archivo Parquet en disco: {tamaño_archivo_mb:.2f} MB")
print(f"Memoria RAM del DataFrame en Pandas: {memoria_dataframe_mb:.2f} MB")

Tiempos de lectura (s): [0.2025, 0.1829, 0.1926]
Mediana tiempo de lectura: 0.1926 segundos

Tamaño del archivo Parquet en disco: 123.83 MB
Memoria RAM del DataFrame en Pandas: 371.93 MB


In [5]:
def medir_tiempo(funcion, repeticiones=3):
    """
    Ejecuta una función N veces, limpia la memoria antes de cada corrida 
    y retorna el resultado final junto con la mediana de tiempos.
    """
    tiempos = []
    resultado = None
    for i in range(repeticiones):
        gc.collect()
        inicio = time.perf_counter()
        resultado = funcion()
        fin = time.perf_counter()
        tiempos.append(fin - inicio)
        if i < repeticiones - 1:
            del resultado
    return resultado, tiempos, float(np.median(tiempos))

print("Función 'medir_tiempo' inicializada.")

Función 'medir_tiempo' inicializada.


In [6]:
resultados_benchmark = []

# ==========================================
# A. DATASHADER (Rejilla 100x100)
# ==========================================
def ejecutar_datashader():
    canvas = ds.Canvas(
        plot_width=100,
        plot_height=100,
        x_range=(df["edad"].min(), df["edad"].max()),
        y_range=(df["altura"].min(), df["altura"].max())
    )
    return canvas.points(df, "edad", "altura", agg=ds.count())

agg, tiempos_ds, mediana_ds = medir_tiempo(ejecutar_datashader)
array_ds = np.asarray(agg.values)
np.savez_compressed("../resultados/representaciones/datashader_100x100.npz", conteos=array_ds)

resultados_benchmark.append({
    "Herramienta": "Datashader",
    "Operación": "Cuadrícula 100 × 100",
    "Tiempo mediano (s)": mediana_ds,
    "Memoria salida (MB)": array_ds.nbytes / (1024 ** 2),
    "Archivo": "datashader_100x100.npz"
})
print(f"✓ Datashader evaluado: {mediana_ds:.4f} s | Forma: {agg.shape}")

# ==========================================
# B. VAEX (Binning 100x100)
# ==========================================
df_vaex = vaex.open(RUTA)
edad_min, edad_max = float(df["edad"].min()), float(df["edad"].max())
altura_min, altura_max = float(df["altura"].min()), float(df["altura"].max())

def ejecutar_vaex():
    return df_vaex.count(
        binby=["edad", "altura"],
        limits=[[edad_min, edad_max], [altura_min, altura_max]],
        shape=100
    )

resultado_vaex, tiempos_vaex, mediana_vaex = medir_tiempo(ejecutar_vaex)
array_vaex = np.asarray(resultado_vaex)
np.savez_compressed("../resultados/representaciones/vaex_100x100.npz", conteos=array_vaex)

resultados_benchmark.append({
    "Herramienta": "Vaex",
    "Operación": "Cuadrícula 100 × 100",
    "Tiempo mediano (s)": mediana_vaex,
    "Memoria salida (MB)": array_vaex.nbytes / (1024 ** 2),
    "Archivo": "vaex_100x100.npz"
})
print(f"✓ Vaex evaluado: {mediana_vaex:.4f} s | Forma: {array_vaex.shape}")

# ==========================================
# C. DUCKDB (Deciles mediante SQL)
# ==========================================
con = duckdb.connect()
def ejecutar_duckdb():
    return con.execute("""
        WITH datos AS (
            SELECT
                edad,
                NTILE(10) OVER (ORDER BY edad) AS decil
            FROM read_parquet('../data/dataset_5M.parquet')
        )
        SELECT
            decil,
            COUNT(*) AS cantidad,
            MIN(edad) AS minimo,
            MAX(edad) AS maximo,
            AVG(edad) AS media,
            STDDEV_POP(edad) AS desviacion
        FROM datos
        GROUP BY decil
        ORDER BY decil
    """).fetchdf()

deciles, tiempos_duckdb, mediana_duckdb = medir_tiempo(ejecutar_duckdb)
deciles.to_parquet("../resultados/representaciones/duckdb_deciles.parquet", index=False)

resultados_benchmark.append({
    "Herramienta": "DuckDB",
    "Operación": "Resumen de 10 deciles",
    "Tiempo mediano (s)": mediana_duckdb,
    "Memoria salida (MB)": deciles.memory_usage(deep=True).sum() / (1024 ** 2),
    "Archivo": "duckdb_deciles.parquet"
})
print(f"✓ DuckDB evaluado: {mediana_duckdb:.4f} s | Filas: {len(deciles)}")

# ==========================================
# D. POLARS (Agregación y Conversión de Tipos)
# ==========================================
df_polars = pl.read_parquet(RUTA)

# D.1 Agregación por Categoría
def ejecutar_polars_agregacion():
    return (
        df_polars
        .group_by("categoria")
        .agg([
            pl.len().alias("cantidad"),
            pl.col("edad").mean().alias("edad_media"),
            pl.col("ingresos").mean().alias("ingresos_media")
        ])
        .sort("categoria")
    )

resumen_polars, tiempos_polars, mediana_polars = medir_tiempo(ejecutar_polars_agregacion)
resumen_polars.write_parquet("../resultados/representaciones/polars_categorias.parquet")

resultados_benchmark.append({
    "Herramienta": "Polars",
    "Operación": "Agregación por categoría",
    "Tiempo mediano (s)": mediana_polars,
    "Memoria salida (MB)": resumen_polars.estimated_size(unit="mb"),
    "Archivo": "polars_categorias.parquet"
})

# D.2 Conversión de Precisión Float64 -> Float32
def ejecutar_float32():
    return df_polars.with_columns([
        pl.col("edad").cast(pl.Float32),
        pl.col("ingresos").cast(pl.Float32),
        pl.col("altura").cast(pl.Float32)
    ])

df_float32, tiempos_float32, mediana_float32 = medir_tiempo(ejecutar_float32)
df_float32.write_parquet("../resultados/representaciones/polars_float32.parquet")

resultados_benchmark.append({
    "Herramienta": "Polars",
    "Operación": "Conversión a Float32",
    "Tiempo mediano (s)": mediana_float32,
    "Memoria salida (MB)": df_float32.estimated_size(unit="mb"),
    "Archivo": "polars_float32.parquet"
})
print(f"✓ Polars evaluado (Agregación: {mediana_polars:.4f} s | Float32: {mediana_float32:.4f} s)")

# ==========================================
# E. SCIKIT-LEARN (Discretización KBinsDiscretizer)
# ==========================================
X = df[["edad"]].to_numpy()

def ejecutar_sklearn():
    discretizador = KBinsDiscretizer(
        n_bins=10,
        encode="ordinal",
        strategy="quantile",
        random_state=42
    )
    return discretizador.fit_transform(X)

X_bins, tiempos_sklearn, mediana_sklearn = medir_tiempo(ejecutar_sklearn)
df_bins = pd.DataFrame({"edad_bin": X_bins[:, 0].astype(np.int8)})
df_bins.to_parquet("../resultados/representaciones/sklearn_edad_bins.parquet", index=False)

resultados_benchmark.append({
    "Herramienta": "Scikit-Learn",
    "Operación": "Discretización de edad",
    "Tiempo mediano (s)": mediana_sklearn,
    "Memoria salida (MB)": X_bins.nbytes / (1024 ** 2),
    "Archivo": "sklearn_edad_bins.parquet"
})
print(f"✓ Scikit-Learn evaluado: {mediana_sklearn:.4f} s | Salida: {X_bins.shape}")

✓ Datashader evaluado: 0.0695 s | Forma: (100, 100)
✓ Vaex evaluado: 0.0530 s | Forma: (100, 100)
✓ DuckDB evaluado: 0.2954 s | Filas: 10
✓ Polars evaluado (Agregación: 0.0224 s | Float32: 0.0167 s)
✓ Scikit-Learn evaluado: 0.1276 s | Salida: (5000000, 1)


In [7]:
# 1. Medición de tamaño en disco de las representaciones generadas
df_benchmark = pd.DataFrame(resultados_benchmark)

def obtener_tamaño_disco(row):
    ruta = os.path.join("../resultados/representaciones/", row["Archivo"])
    return os.path.getsize(ruta) / (1024 ** 2)

df_benchmark["Tamaño de salida (MB)"] = df_benchmark.apply(obtener_tamaño_disco, axis=1)

# Reorganizar y formatear la tabla final
columnas_ordenadas = [
    "Herramienta", 
    "Operación", 
    "Tiempo mediano (s)", 
    "Tamaño de salida (MB)", 
    "Memoria salida (MB)"
]
tabla_comparativa = df_benchmark[columnas_ordenadas].copy()

# Guardar la tabla de tamaños de las representaciones
tabla_tamaños = df_benchmark[["Archivo", "Tamaño de salida (MB)"]].rename(
    columns={"Tamaño de salida (MB)": "tamaño_mb"}
)
tabla_tamaños.to_csv("../resultados/tablas/tamanos_representaciones.csv", index=False)

# Guardar la tabla comparativa general de rendimiento
tabla_comparativa.to_csv("../resultados/tablas/tabla_rendimiento_comparativo.csv", index=False)

print("=== TABLA COMPARATIVA DE RENDIMIENTO Y REPRESENTACIONES ===")
tabla_comparativa

=== TABLA COMPARATIVA DE RENDIMIENTO Y REPRESENTACIONES ===


,Herramienta,Operación,Tiempo mediano (s),Tamaño de salida (MB),Memoria salida (MB)
0,Datashader,Cuadrícula 100 × 100,0.069530,0.010314,0.038147
1,Vaex,Cuadrícula 100 × 100,0.053009,0.010868,0.076294
2,DuckDB,Resumen de 10 deciles,0.295369,0.004107,0.000584
3,Polars,Agregación por categoría,0.022386,0.001504,0.000100
4,Polars,Conversión a Float32,0.016657,54.167783,81.062317
5,Scikit-Learn,Discretización de edad,0.127562,2.395841,38.146973
